# Exploration du flux GTFS-RT SNCF (transport.data.gouv.fr)

Contexte : `api.sncf.com` (Navitia) ne reflète pas de façon fiable le temps réel pour certains trains régionaux (ex. réseau `liO`, Occitanie) — un retard confirmé sur le site SNCF n'apparaissait pas dans la réponse `departures` (`data_freshness` restait `base_schedule`). Hypothèse : la couverture temps réel de `api.sncf.com` varie selon la région, et le flux GTFS-RT national de transport.data.gouv.fr pourrait être plus complet.

Source : https://transport.data.gouv.fr/datasets/horaires-sncf

Flux Trip Updates : https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-trip-updates (mis à jour toutes les 2 minutes, trains circulant dans les 60 prochaines minutes).

In [1]:
%load_ext autoreload
%autoreload 2

import requests
from google.transit import gtfs_realtime_pb2

GTFS_RT_TRIP_UPDATES_URL = "https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-trip-updates"

## 1. Récupérer et décoder le flux Trip Updates

Format protobuf binaire, pas JSON — utiliser `.content` sur la réponse, pas `.json()`. Vérifier au passage si le endpoint demande une authentification (401/403 sans clé ?).

In [11]:
from google.transit import gtfs_realtime_pb2
import requests

response = requests.get("https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-trip-updates")
feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

1629957

## 2. Retrouver le train 876213 dans le flux

Ce train (K13, réseau `liO`, Toulouse Montaudran → Toulouse Matabiau, le 2026-09-22) était en retard confirmé sur le site SNCF, mais `api.sncf.com` le montrait en `base_schedule` sans écart. Identifier sous quel format le `trip_id` apparaît ici (numéro brut, id plus long...) — à observer, pas à deviner.

In [12]:
for entity in feed.entity:
    if entity.HasField('trip_update'):
        trip_id = entity.trip_update.trip.trip_id
        if "876213" in trip_id:
            print(entity)

id: "OCESN876213F1187_F:TER:FR:Line::1CEA14EA-7F5A-4D44-A739-F6A13138F139::87775007:87611004:22:1904:20261211"
trip_update {
  trip {
    trip_id: "OCESN876213F1187_F:TER:FR:Line::1CEA14EA-7F5A-4D44-A739-F6A13138F139::87775007:87611004:22:1904:20261211"
    start_time: "15:36:00"
    start_date: "20260922"
  }
  stop_time_update {
    departure {
      delay: 0
      time: 1790084160
    }
    stop_id: "StopPoint:OCETrain TER-87775007"
  }
  stop_time_update {
    arrival {
      delay: 0
      time: 1790084700
    }
    departure {
      delay: 0
      time: 1790084760
    }
    stop_id: "StopPoint:OCETrain TER-87775114"
  }
  stop_time_update {
    arrival {
      delay: 0
      time: 1790085120
    }
    departure {
      delay: 0
      time: 1790085240
    }
    stop_id: "StopPoint:OCETrain TER-87773408"
  }
  stop_time_update {
    arrival {
      delay: 0
      time: 1790085600
    }
    departure {
      delay: 0
      time: 1790085660
    }
    stop_id: "StopPoint:OCETrain TER-

## 3. Comparer le retard reporté ici à ce qu'observait `api.sncf.com`

Si ce flux montre le vrai retard là où `api.sncf.com` ne le montrait pas : ça confirme un vrai trou de couverture régionale sur `api.sncf.com`, et GTFS-RT devient une source à considérer pour au moins une partie du corridor.

## 4. Explorer le flux SIRI ET Lite (quai/voie)

Standard SIRI, champs `ArrivalPlatformName`/`DeparturePlatformName` prévus pour ça dans le profil Estimated Timetable. Vérifier si SNCF les remplit réellement, sur le même train 876213 si possible pour rester cohérent avec ce qu'on a déjà observé.

Flux : https://proxy.transport.data.gouv.fr/resource/sncf-siri-lite-estimated-timetable (marqué "version Béta" par SNCF). Format à vérifier (XML SIRI classique, ou variante JSON "Lite" simplifiée) — ne pas supposer avant d'avoir regardé une vraie réponse.

In [7]:
from google.transit import gtfs_realtime_pb2
import requests
import xmltodict
import json

response = requests.get("https://proxy.transport.data.gouv.fr/resource/sncf-siri-lite-estimated-timetable")
response_as_dict = xmltodict.parse(response.content)
with open('scrap.json', 'w+') as fh:
    json.dump(response_as_dict, fh, indent=2, ensure_ascii=False)


## 5. Vérifier si `ArrivalPlatformName`/`DeparturePlatformName` sont renseignés

Le standard permet ces champs, rien ne garantit que SNCF les remplit (même prudence que pour `stop_time_properties` côté GTFS-RT). Chercher sur plusieurs trains, en particulier un train qui a eu une perturbation — c'est là qu'un changement de quai a le plus de chances d'apparaître si la donnée existe.

In [10]:
response = requests.get("https://proxy.transport.data.gouv.fr/resource/sncf-siri-lite-estimated-timetable")
response_as_dict = xmltodict.parse(response.content)
arrival_plateforme_name = response_as_dict['Siri']['ServiceDelivery']['EstimatedTimetableDelivery']['EstimatedJourneyVersionFrame']['EstimatedVehicleJourney'][0]['RecordedCalls']['RecordedCall'][0]['DeparturePlatformName']

print(arrival_plateforme_name)

4
